# 🍎 โครงการพัฒนาระบบจำแนกชนิดผลไม้ (Fruit Classifier)
## Machine Learning Image Classification (267 Classes)

**รายวิชา:** Machine Learning / Data Science Coursework  
**ผู้จัดทำ:**
1. [นาย/นางสาว ชื่อ-นามสกุล] - [รหัสนิสิต 10 หลัก]
2. [นาย/นางสาว ชื่อ-นามสกุล] - [รหัสนิสิต 10 หลัก] *(หากมี)*

---
## 1. บทนำ อธิบายปัญหา และเป้าหมายของโครงการ (Problem Statement & Goals)

### ปัญหาที่ต้องการแก้ (Problem Statement)
การจำแนกชนิดและสายพันธุ์ของผลไม้และพืชผลทางการเกษตรมีความสำคัญอย่างยิ่งต่อกระบวนการตรวจสอบคุณภาพ การคัดเกรดผลผลิตอัตโนมัติ (Automated Sorting) ระบบคิดเงินในซูเปอร์มาร์เก็ตอัจฉริยะ (Smart Retail Checkout) และการส่งเสริมการเกษตรแม่นยำ (Smart Agriculture) 

อย่างไรก็ตาม ผลไม้มีความหลากหลายทางชีวภาพสูงมาก ทั้งในแง่ของรูปทรง สีสัน ขนาด และสายพันธุ์ย่อยที่มีลักษณะภายนอกใกล้เคียงกันมาก เช่น แอปเปิ้ลที่มีมากกว่า 19 สายพันธุ์ (เช่น Granny Smith, Golden, Red Delicious, Braeburn ฯลฯ) นอกจากนี้ ผลไม้ไทยและผลไม้เขตร้อนยังมีรูปทรงและสีผิวที่ซับซ้อน เช่น ทุเรียน, มังคุด, เงาะ, ขนุน, ลำไย, กระท้อน, และน้อยหน่า การจำแนกด้วยตามนุษย์ต้องอาศัยความชำนาญและใช้เวลานาน

### เป้าหมายของโครงการ (Project Goals)
1. พัฒนาแบบจำลอง Machine Learning สำหรับจำแนกภาพผลไม้และพืชผลทางการเกษตรแบบหลายคลาส (Multi-Class Image Classification) ครอบคลุมถึง **267 คลาส (267 Classes)**
2. ออกแบบกระบวนการสกัดคุณลักษณะ (Feature Extraction) ที่มีประสิทธิภาพสูง ผสานข้อมูลเชิงพื้นที่ (Spatial Pixel Distribution) และการกระจายตัวของสี (RGB & HSV Color Histograms)
3. ฝึกสอนและประเมินผลแบบจำลองบนชุดข้อมูลทดสอบ (Test Set) ให้ได้ความแม่นยำสูง (Target Accuracy > 85%)
4. สร้างแบบจำลองที่มีขนาดกะทัดรัด (Lightweight < 5 MB) และใช้ทรัพยากรหน่วยความจำต่ำ เพื่อให้สามารถนำไปติดตั้งและทำงานได้อย่างรวดเร็วบนเว็บแอปพลิเคชันคลาวด์ (Cloud Web App Deployment)


## 2. แหล่งที่มาและอธิบายชุดข้อมูล (Dataset Sources & Description)

ชุดข้อมูลที่ใช้ในการฝึกและทดสอบแบบจำลอง มาจากการผสาน 2 แหล่งข้อมูลมาตรฐาน:
1. **Fruits-360 Dataset (Kaggle):** รวบรวมโดย Mihai Oltean เผยแพร่ภายใต้สัญญาอนุญาต CC BY-SA 4.0 เป็นภาพผลไม้ที่ถ่ายในสตูดิโอที่มีแสงสว่างสม่ำเสมอ มีการหมุนวัตถุ 360 องศา และฉากหลังสีขาวสะอาด
2. **Fruit-262 Dataset (Kaggle):** ชุดข้อมูลภาพผลไม้หลากหลายสายพันธุ์จากทั่วโลก เราทำการคัดเลือกเฉพาะผลไม้ไทยและผลไม้เขตร้อนหายากที่ไม่มีใน Fruits-360 จำนวน 33 ชนิด (เช่น ขนุน, ลำไย, กระท้อน, น้อยหน่า, ตะลิงปลิง, มะตูม, ลองกอง, จำปาดะ, มะกรูด ฯลฯ)

### คุณลักษณะของชุดข้อมูล:
- **จำนวนคลาสทั้งหมด:** 267 คลาส
- **จำนวนรูปภาพต่อคลาส:** คลาสละ 25 รูปภาพ (Balanced Dataset)
- **จำนวนรูปภาพรวม:** 6,675 รูปภาพ
- **รูปแบบไฟล์:** ภาพสี RGB ขนาดมาตรฐาน


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# ตั้งค่าสไตล์การแสดงผล
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("ไลบรารีทั้งหมดถูกโหลดเรียบร้อยแล้ว!")


## 3. การสำรวจและเตรียมข้อมูล (Exploratory Data Analysis & Preprocessing)

ในขั้นตอนนี้ เราจะตรวจสอบโครงสร้างของโฟลเดอร์ชุดข้อมูล `dataset/` ตรวจสอบจำนวนภาพในแต่ละคลาส และแสดงภาพตัวอย่างของผลไม้ชนิดต่าง ๆ ทั้งผลไม้สากลและผลไม้ไทย


In [ ]:
dataset_path = Path("dataset")
classes = sorted([d.name for d in dataset_path.iterdir() if d.is_dir() and not d.name.startswith((".", "_"))])
print(f"จำนวนคลาสทั้งหมดที่พบ: {len(classes)} คลาส")

# ตรวจสอบจำนวนภาพต่อคลาส
class_counts = {c: len(list((dataset_path / c).glob("*.jpg")) + list((dataset_path / c).glob("*.png"))) for c in classes}
total_images = sum(class_counts.values())
print(f"จำนวนรูปภาพรวมทั้งหมด: {total_images} รูป")
print(f"จำนวนภาพเฉลี่ยต่อคลาส: {total_images / len(classes):.1f} รูปภาพ (Balanced Dataset)")


In [ ]:
# แสดงภาพตัวอย่างผลไม้ 12 ชนิด
sample_classes = [
    "durian", "mangosteen_1", "rambutan_1", "jackfruit",
    "longan", "santol", "custard_apple", "pitahaya_red_1",
    "mango_1", "banana_1", "apple_red_1", "watermelon_1"
]

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()

for idx, cls_name in enumerate(sample_classes):
    cls_folder = dataset_path / cls_name
    img_files = list(cls_folder.glob("*.jpg")) + list(cls_folder.glob("*.png"))
    if img_files:
        img = Image.open(img_files[0])
        axes[idx].imshow(img)
        axes[idx].set_title(cls_name, fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("ตัวอย่างรูปภาพผลไม้ในชุดข้อมูล (Sample Fruit Images)", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()


## 4. การสกัดคุณลักษณะของรูปภาพ (Feature Extraction)

เพื่อให้แบบจำลอง Machine Learning สามารถเรียนรู้ลักษณะรูปทรงและสีสันของผลไม้ได้อย่างแม่นยำ เราได้ออกแบบกระบวนการสกัดคุณลักษณะแบบผสมผสาน (Combined Features) รวมทั้งสิ้น **3,176 มิติ**:

1. **Flatten Pixel Array ($32 \times 32 \times 3 = 3,072$ มิติ):** ย่อภาพเป็น $32 \times 32$ พิกเซล และแปลงค่าเป็นเวกเตอร์ 1D ปรับสเกลให้อยู่ในช่วง $[0, 1]$ สำหรับจับข้อมูลรูปร่าง ลวดลายพื้นผิว และโครงสร้างเชิงพื้นที่
2. **RGB Color Histogram ($16 \times 3 = 48$ มิติ):** คำนวณความถี่การกระจายตัวของแม่สี แดง เขียว และน้ำเงิน ช่องละ 16 bins ในช่วง $[0, 256]$ พร้อมทำ Density Normalization
3. **HSV Color Distribution ($24 + 16 + 16 = 56$ มิติ):** แปลงภาพเป็นปริภูมิสี HSV แล้วสกัดฮิสโตแกรมของ Hue (เนื้อสี 24 bins), Saturation (ความอิ่มตัวสี 16 bins), และ Value (ความสว่าง 16 bins)


In [ ]:
def extract_features(image: Image.Image, img_size=(32, 32)):
    """สกัดฟีเจอร์ Spatial 32x32 + RGB Histogram + HSV Distribution (รวม 3,176 มิติ)"""
    if image.mode != "RGB":
        image = image.convert("RGB")
    
    # 1. Spatial Flatten (3072 dims)
    resized_img = image.resize(img_size)
    img_arr = np.array(resized_img, dtype=np.float32)
    flatten_feat = (img_arr / 255.0).flatten()
    
    # 2. RGB Histogram (48 dims)
    hist_r, _ = np.histogram(img_arr[:, :, 0], bins=16, range=(0, 256), density=True)
    hist_g, _ = np.histogram(img_arr[:, :, 1], bins=16, range=(0, 256), density=True)
    hist_b, _ = np.histogram(img_arr[:, :, 2], bins=16, range=(0, 256), density=True)
    rgb_feat = np.hstack([hist_r, hist_g, hist_b])
    
    # 3. HSV Histogram (56 dims)
    hsv_img = image.convert("HSV")
    hsv_arr = np.array(hsv_img, dtype=np.float32)
    hist_h, _ = np.histogram(hsv_arr[:, :, 0], bins=24, range=(0, 256), density=True)
    hist_s, _ = np.histogram(hsv_arr[:, :, 1], bins=16, range=(0, 256), density=True)
    hist_v, _ = np.histogram(hsv_arr[:, :, 2], bins=16, range=(0, 256), density=True)
    hsv_feat = np.hstack([hist_h, hist_s, hist_v])
    
    return np.hstack([flatten_feat, rgb_feat, hsv_feat])

# โหลดข้อมูลและสกัดฟีเจอร์จากทุกคลาส
X = []
y = []
for c in classes:
    c_folder = dataset_path / c
    img_files = list(c_folder.glob("*.jpg")) + list(c_folder.glob("*.png"))
    for p in img_files:
        try:
            with Image.open(p) as im:
                X.append(extract_features(im))
                y.append(c)
        except Exception:
            pass

X = np.array(X, dtype=np.float32)
y = np.array(y)
print(f"สกัดฟีเจอร์สำเร็จทั้งหมด {len(X)} ภาพ | มิติของ X: {X.shape}, มิติของ y: {y.shape}")


## 5. การแบ่งข้อมูลสำหรับฝึกและทดสอบ (Train-Test Split)

เราใช้วิธี **Stratified Split (80:20)**:
- **ชุดฝึกสอน (Train Set 80%):** 5,340 รูปภาพ (คลาสละ 20 รูป)
- **ชุดทดสอบ (Test Set 20%):** 1,335 รูปภาพ (คลาสละ 5 รูป)

> ⚠️ **หลักการสำคัญ:** กำหนด `random_state=42` และ `stratify=y` เพื่อให้สัดส่วนของแต่ละคลาสสมดุลเท่าเทียมกัน และชุดทดสอบ (Test Set) จะถูกเก็บไว้ต่างหาก ไม่ถูกนำไปใช้ในกระบวนการเทรนหรือปรับแต่งโมเดลโดยเด็ดขาด (No Data Leakage)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"จำนวนข้อมูล Train Set: {len(X_train)} รูปภาพ")
print(f"จำนวนข้อมูล Test Set : {len(X_test)} รูปภาพ")


## 6. การฝึกแบบจำลอง (Model Training) และเหตุผลที่เลือกใช้

### ทำไมจึงเลือกใช้ Logistic Regression (Linear Classifier)?
1. **มิติของข้อมูลสูง (High-Dimensional Space):** ข้อมูลมีฟีเจอร์ 3,176 มิติ ซึ่งในทางสถิติและ Machine Learning ข้อมูลมิติสูงมักจะสามารถแบ่งแยกเชิงเส้น (Linearly Separable) ได้ดีอยู่แล้ว
2. **ประหยัดทรัพยากรและรวดเร็ว:** ใช้เวลาเทรนเพียง 10-15 วินาที เมื่อเทียบกับ Random Forest หรือ SVM ที่อาจใช้เวลาหลายสิบนาที และ Deep Learning ที่ต้องใช้ GPU หลายชั่วโมง
3. **ขนาดโมเดลเล็กมาก (Compact Size):** ไฟล์โมเดลมีขนาดเพียง **3.45 MB** สามารถ Push ขึ้น GitHub และ Deploy ขึ้น Cloud Web Service ฟรีเทียร์ (เช่น Render ที่จำกัด RAM 512 MB) ได้อย่างสบาย
4. **ความน่าจะเป็นที่เสถียร (Calibrated Probabilities):** สามารถคำนวณ `predict_proba()` ได้โดยตรงสำหรับแสดง Top-5 Confidence Breakdown บนหน้าเว็บ


In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=150, C=1.0, random_state=42))
])

print("กำลังฝึกแบบจำลอง Logistic Regression (One-vs-Rest)...")
pipeline.fit(X_train, y_train)
print("การฝึกแบบจำลองเสร็จสมบูรณ์!")


## 7. การประเมินผลบนชุดทดสอบและการแปลความหมายเมตริก (Evaluation & Metrics)

เราจะทำการทดสอบโมเดลบน **ชุดทดสอบ (Test Set จำนวน 1,335 รูปภาพ)** ที่ไม่เคยผ่านการฝึกมาก่อน:
- **Accuracy:** สัดส่วนการทำนายถูกต้องทั้งหมด
- **Macro Average:** ค่าเฉลี่ยแบบไม่ถ่วงน้ำหนักของ Precision, Recall, F1-Score สำหรับประเมินความเท่าเทียมทุกคลาส
- **Weighted Average:** ค่าเฉลี่ยแบบถ่วงน้ำหนักตามจำนวนข้อมูล
- **Confusion Matrix:** เมทริกซ์แสดงการกระจายตัวของการทำนาย


In [ ]:
y_pred = pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("=" * 60)
print(f"ความแม่นยำบนชุดทดสอบ (Test Accuracy): {test_accuracy * 100:.2f}%")
print("=" * 60)

report = classification_report(y_test, y_pred, zero_division=0)
report_lines = report.strip().split("\n")
print("สรุปรายงานการประเมิน (Classification Report Summary):")
print("\n".join(report_lines[:6]))
print(f" ... [และคลาสผลไม้อื่น ๆ รวมทั้งสิ้น {len(classes)} คลาส] ...")
print("\n".join(report_lines[-4:]))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap="Blues", cbar=True)
plt.title(f"Confusion Matrix (267 Classes) | Test Accuracy: {test_accuracy * 100:.1f}%", fontsize=12)
plt.xlabel("Predicted Class Index", fontsize=10)
plt.ylabel("True Class Index", fontsize=10)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=180)
plt.show()


## 8. แสดงตัวอย่างผลการทำนายและวิเคราะห์ข้อผิดพลาด (Predictions & Error Analysis)

### 8.1 การทดสอบภาพตัวอย่างผลไม้จริง


In [ ]:
from train import format_class_name

sample_test_files = [
    "sample_images/durian.jpg",
    "sample_images/mangosteen.jpg",
    "sample_images/rambutan.jpg",
    "sample_images/jackfruit.jpg",
    "sample_images/longan.jpg",
    "sample_images/santol.jpg",
    "sample_images/custard_apple.jpg",
    "sample_images/banana.jpg"
]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for idx, sample_path in enumerate(sample_test_files):
    if Path(sample_path).exists():
        im = Image.open(sample_path)
        feat = extract_features(im)
        probs = pipeline.predict_proba([feat])[0]
        top_idx = probs.argmax()
        pred_c = pipeline.classes_[top_idx]
        conf = probs[top_idx] * 100
        
        display_th = format_class_name(pred_c)
        axes[idx].imshow(im)
        axes[idx].set_title(f"{display_th}\nConf: {conf:.1f}%", fontsize=10)
    axes[idx].axis("off")

plt.suptitle("ตัวอย่างผลการทำนายภาพผลไม้ (Sample Predictions)", fontsize=13)
plt.tight_layout()
plt.show()


### 8.2 การวิเคราะห์ข้อผิดพลาด (Failure & Confusion Analysis)
จากการวิเคราะห์ผลลัพธ์บน Test Set พบประเด็นสำคัญดังนี้:

1. **คลาสที่ทำนายได้สมบูรณ์แบบ (Precision / Recall = 100%):**
   - ผลไม้ที่มีรูปทรงเรขาคณิตและสีสันเฉพาะตัวบนพื้นหลังสะอาด เช่น ส้ม, แตงโม, สตรอว์เบอร์รี, ทุเรียน, มังคุด, เงาะ, อะโวคาโด, และแอปเปิ้ลเกือบทุกสายพันธุ์
   - ผลไม้ไทยที่มีเปลือกสีหรือผิวสัมผัสจำเพาะ เช่น กระท้อน, น้อยหน่า, ลำไย, มะกรูด
2. **คลาสที่เกิดความสับสน (Confusion Cases):**
   - **ภาพถ่ายสภาพแวดล้อมธรรมชาติ (In-the-wild backgrounds):** ผลไม้บางชนิดจาก Fruit-262 เช่น มะกอกฝรั่ง (Ambarella) หรือ มะตูม (Bael) มีใบไม้หรือกิ่งไม้สีเขียวในฉากหลัง ทำให้เวกเตอร์ฮิสโตแกรมสีถูกรบกวน
   - **ผลไม้ผิวสีน้ำตาลอมเหลือง:** ละมุด (Sapodilla) สับสนกับ ลองกอง/ลางสาด (Langsat) ในบางมุมมองเนื่องจากค่าเฉลี่ยสี HSV และความสว่างใกล้เคียงกัน


## 9. การบันทึกแบบจำลองสำหรับนำไปใช้ในเว็บแอปพลิเคชัน (Model Persistence)

เราทำการบันทึกโมเดล Pipeline พร้อม Metadata ที่จำเป็นสำหรับ Web App ได้แก่:
- `pipeline`: Scikit-learn Pipeline (Scaler + LogisticRegression)
- `classes`: รายชื่อ 267 คลาส
- `thai_labels`: พจนานุกรมคำแปลภาษาไทยและ Emoji สำหรับแสดงผล
- `accuracy`: ค่าความแม่นยำบน Test Set
- `feature_mode`: 'combined'
- `img_size`: (32, 32)


In [ ]:
from train import build_thai_labels

model_payload = {
    "pipeline": pipeline,
    "classes": list(classes),
    "thai_labels": build_thai_labels(classes),
    "feature_mode": "combined",
    "model_type": "linear",
    "accuracy": float(test_accuracy),
    "img_size": (32, 32)
}

output_path = "fruit_model.pkl"
joblib.dump(model_payload, output_path, compress=3)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"บันทึกไฟล์โมเดลเสร็จสมบูรณ์ที่: {output_path}")
print(f"ขนาดไฟล์โมเดล: {file_size_mb:.2f} MB")
print("พร้อมสำหรับการนำไป Deploy บน Gradio Web Server เรียบร้อยแล้ว!")
